GOLDEN DATASET

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

Project root: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics
Raw data: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\data\raw
Processed data: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\data\processed


In [2]:
# LOAD COMPANY-PROVIDED DATA

datasets = {}

for file in sorted(RAW_DIR.glob("*.csv")):
    datasets[file.stem] = pd.read_csv(
        file,
        low_memory=False
    )

print(f"Datasets loaded: {len(datasets)}")

for name, df in datasets.items():
    print(
        f"{name}: "
        f"{len(df):,} rows × {len(df.columns)} columns"
    )

Datasets loaded: 18
account_status_history: 60,000 rows × 8 columns
accounts: 30,000 rows × 11 columns
agent_sessions: 15,000 rows × 7 columns
agents: 30,000 rows × 8 columns
borrowers: 30,600 rows × 8 columns
call_attempts: 120,000 rows × 9 columns
call_dispositions: 35,000 rows × 8 columns
calls: 91,350 rows × 11 columns
campaigns: 120 rows × 7 columns
complaints: 8,000 rows × 9 columns
daily_targeting: 45,000 rows × 7 columns
data_dictionary: 143 rows × 3 columns
field_visits: 25,000 rows × 10 columns
payments: 25,500 rows × 9 columns
promises_to_pay: 18,000 rows × 9 columns
sms_events: 45,000 rows × 8 columns
vendor_telephony: 15 rows × 6 columns
whatsapp_events: 60,600 rows × 8 columns


In [3]:
#PAYMENT GOLDEN DATASET — STARTING POINT
payments_raw = datasets["payments"].copy()

print("Raw payment records:", len(payments_raw))
print(
    "Raw successful payment records:",
    (payments_raw["payment_status"] == "SUCCESS").sum()
)

display(payments_raw.head())

Raw payment records: 25500
Raw successful payment records: 17880


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
0,PAYMENT0000001,ACC0015539,BRW0011363,2026-02-27 01:28:12,TXN0000007457,22433.23,FAILED,CARD,VND0000001
1,PAYMENT0000002,ACC0004445,BRW0011306,2026-07-23 20:25:20,TXN0000030016,23295.11,FAILED,UPI,VND0000004
2,PAYMENT0000003,ACC0003853,BRW0002549,2026-01-11 21:27:46,TXN0000010466,118814.97,FAILED,NACH,VND0000013
3,PAYMENT0000004,ACC0017531,BRW0006618,2026-06-16 02:35:21,TXN0000031781,145082.17,REVERSED,CASH,VND0000013
4,PAYMENT0000005,ACC0001770,BRW0011492,2026-03-03 06:08:23,TXN0000059257,115148.46,SUCCESS,NETBANKING,VND0000013


In [4]:
# PAYMENT QUALITY FLAGS
payments_golden = payments_raw.copy()

payments_golden["quality_flag"] = "VALID"

# Exact duplicate rows
exact_duplicate_mask = payments_golden.duplicated(
    keep=False
)

payments_golden.loc[
    exact_duplicate_mask,
    "quality_flag"
] = "EXACT_DUPLICATE"

# Missing payment reference
missing_reference_mask = (
    payments_golden["payment_reference"].isna()
)

payments_golden.loc[
    missing_reference_mask
    & (payments_golden["quality_flag"] == "VALID"),
    "quality_flag"
] = "MISSING_REFERENCE"

payments_golden["quality_flag"].value_counts(
    dropna=False
)

quality_flag
VALID                24146
EXACT_DUPLICATE        972
MISSING_REFERENCE      382
Name: count, dtype: int64

In [5]:
# GOLDEN PAYMENT TABLE — EXACT DUPLICATE EXCLUSION
payments_golden_clean = payments_golden[
    payments_golden["quality_flag"] != "EXACT_DUPLICATE"
].copy()

print(
    "Raw payment records:",
    len(payments_golden)
)

print(
    "Golden payment records:",
    len(payments_golden_clean)
)

print(
    "Excluded exact duplicate records:",
    (
        payments_golden["quality_flag"]
        == "EXACT_DUPLICATE"
    ).sum()
)

Raw payment records: 25500
Golden payment records: 24528
Excluded exact duplicate records: 972


In [6]:
#SAVE PAYMENT GOLDEN DATASET
payment_output = (
    PROCESSED_DIR
    / "golden_payments.csv"
)

payments_golden_clean.to_csv(
    payment_output,
    index=False
)

print(
    "Saved:",
    payment_output
)

Saved: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\data\processed\golden_payments.csv


In [7]:
#GOLDEN ACCOUNTS — LOAD SOURCE TABLES
accounts_raw = datasets["accounts"].copy()
status_history_raw = datasets["account_status_history"].copy()

print("Accounts:", len(accounts_raw))
print("Account status history:", len(status_history_raw))

display(accounts_raw.head())
display(status_history_raw.head())

Accounts: 30000
Account status history: 60000


,account_id,borrower_id,loan_type,principal_amount,outstanding_amount,dpd,risk_segment,status,opened_at,timezone,schema_version
0,ACC0000001,BRW0010742,CONSUMER,603443.40,678074.03,15,MEDIUM,CLOSED,2025-11-11 04:37:00,Asia/Dubai,v1
1,ACC0000002,BRW0009382,BNPL,277190.14,464893.44,5,LOW,ACTIVE,2025-11-13 15:59:44,UTC,v3
2,ACC0000003,BRW0003966,CREDIT_CARD,628658.43,22565.37,60,NPA,WRITEOFF,2025-09-12 04:59:20,Asia/Dubai,v2
3,ACC0000004,BRW0001993,PERSONAL,76435.07,508427.73,30,LOW,CLOSED,2025-05-25 19:29:38,UTC,v2
4,ACC0000005,BRW0009976,CONSUMER,646797.08,563858.01,180,LOW,WRITEOFF,2025-09-07 14:59:13,Asia/Dubai,v2


,history_id,account_id,borrower_id,event_at,status,changed_by,source,recorded_at
0,HISTORY0000001,ACC0004322,BRW0005490,2026-07-03 04:28:38,WRITEOFF,AGT0000039,CALL,2026-07-03 00:41:54
1,HISTORY0000002,ACC0024644,BRW0005343,2026-07-18 02:03:23,DELINQUENT,AGT0000059,CALL,2026-07-18 22:42:02
2,HISTORY0000003,ACC0020233,BRW0003150,2026-04-15 09:17:47,CLOSED,AGT0000080,PAYMENT,2026-04-15 19:40:51
3,HISTORY0000004,ACC0004384,BRW0008483,2026-01-16 10:52:04,WRITEOFF,SYSTEM,CORE,2026-01-16 23:05:28
4,HISTORY0000005,ACC0022967,BRW0009477,2026-05-22 04:13:04,PTP,AGT0000073,CORE,2026-05-22 07:08:28


In [8]:
# ACCOUNT KEY VALIDATION
account_key_summary = pd.DataFrame({
    "metric": [
        "Total account rows",
        "Unique account_id",
        "Duplicate account_id rows",
        "Missing account_id"
    ],
    "value": [
        len(accounts_raw),
        accounts_raw["account_id"].nunique(),
        accounts_raw["account_id"].duplicated().sum(),
        accounts_raw["account_id"].isna().sum()
    ]
})

account_key_summary

,metric,value
0,Total account rows,30000
1,Unique account_id,30000
2,Duplicate account_id rows,0
3,Missing account_id,0


In [9]:
#ACCOUNT → BORROWER ENTITY RESOLUTION



borrowers_raw = datasets["borrowers"].copy()

valid_borrower_ids = set(
    borrowers_raw["borrower_id"]
    .dropna()
    .astype(str)
)

accounts_raw["borrower_id_check"] = (
    accounts_raw["borrower_id"]
    .astype("string")
    .isin(valid_borrower_ids)
)

borrower_resolution_summary = pd.DataFrame({
    "metric": [
        "Total accounts",
        "Accounts with matched borrower_id",
        "Accounts with unmatched borrower_id"
    ],
    "count": [
        len(accounts_raw),
        accounts_raw["borrower_id_check"].sum(),
        (~accounts_raw["borrower_id_check"]).sum()
    ]
})

borrower_resolution_summary["percentage"] = (
    borrower_resolution_summary["count"]
    / len(accounts_raw)
    * 100
)

borrower_resolution_summary


,metric,count,percentage
0,Total accounts,30000,100.00
1,Accounts with matched borrower_id,27087,90.29
2,Accounts with unmatched borrower_id,2913,9.71


In [10]:
# ACCOUNT DATA QUALITY FLAG
accounts_golden = accounts_raw.copy()

accounts_golden["quality_flag"] = "VALID"

accounts_golden.loc[
    ~accounts_golden["borrower_id_check"],
    "quality_flag"
] = "UNRESOLVED_BORROWER_ID"

accounts_golden[
    "quality_flag"
].value_counts(dropna=False)

quality_flag
VALID                     27087
UNRESOLVED_BORROWER_ID     2913
Name: count, dtype: int64

In [11]:
#ACCOUNT STATUS RECONCILIATION



status_history_raw["event_at_dt"] = pd.to_datetime(
    status_history_raw["event_at"],
    errors="coerce"
)

status_history_raw["recorded_at_dt"] = pd.to_datetime(
    status_history_raw["recorded_at"],
    errors="coerce"
)

latest_status = (
    status_history_raw[
        status_history_raw["event_at_dt"].notna()
    ]
    .sort_values(
        ["account_id", "event_at_dt", "recorded_at_dt"]
    )
    .groupby("account_id")
    .tail(1)
    [["account_id", "status"]]
    .rename(
        columns={
            "status": "latest_historical_status"
        }
    )
)

accounts_golden = accounts_golden.merge(
    latest_status,
    on="account_id",
    how="left"
)

accounts_golden["status_history_match"] = (
    accounts_golden["status"]
    == accounts_golden["latest_historical_status"]
)

display(
    accounts_golden[
        [
            "account_id",
            "status",
            "latest_historical_status",
            "status_history_match"
        ]
    ].head(20)
)


,account_id,status,latest_historical_status,status_history_match
0,ACC0000001,CLOSED,DELINQUENT,False
1,ACC0000002,ACTIVE,DELINQUENT,False
2,ACC0000003,WRITEOFF,NPA,False
3,ACC0000004,CLOSED,ACTIVE,False
4,ACC0000005,WRITEOFF,CLOSED,False
5,ACC0000006,CLOSED,PTP,False
6,ACC0000007,CLOSED,DELINQUENT,False
7,ACC0000008,WRITEOFF,CLOSED,False
8,ACC0000009,PAID,PTP,False
9,ACC0000010,ACTIVE,ACTIVE,True


In [12]:
#FINAL ACCOUNT QUALITY CLASSIFICATION
accounts_golden.loc[
    accounts_golden["latest_historical_status"].notna()
    & ~accounts_golden["status_history_match"],
    "quality_flag"
] = "STATUS_MISMATCH"

accounts_golden["quality_flag"].value_counts(
    dropna=False
)

quality_flag
STATUS_MISMATCH           22295
VALID                      6954
UNRESOLVED_BORROWER_ID      751
Name: count, dtype: int64

In [13]:
#SAVE GOLDEN ACCOUNTS
account_output = (
    PROCESSED_DIR / "golden_accounts.csv"
)

accounts_golden.to_csv(
    account_output,
    index=False
)

print("Saved:", account_output)
print("Rows:", len(accounts_golden))

Saved: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\data\processed\golden_accounts.csv
Rows: 30000


In [14]:
# LOAD COLLECTION INTERACTION TABLES
calls_raw = datasets["calls"].copy()
call_attempts_raw = datasets["call_attempts"].copy()
call_dispositions_raw = datasets["call_dispositions"].copy()

print("Calls:", len(calls_raw))
print("Call attempts:", len(call_attempts_raw))
print("Call dispositions:", len(call_dispositions_raw))

print("\nCalls columns:")
print(calls_raw.columns.tolist())

print("\nCall attempts columns:")
print(call_attempts_raw.columns.tolist())

print("\nCall dispositions columns:")
print(call_dispositions_raw.columns.tolist())

Calls: 91350
Call attempts: 120000
Call dispositions: 35000

Calls columns:
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone']

Call attempts columns:
['attempt_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'attempt_no', 'vendor_id', 'attempt_status']

Call dispositions columns:
['disposition_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'disposition_code', 'disposition_version']


In [15]:
#CALL KEY VALIDATION

call_key_summary = pd.DataFrame({
    "metric": [
        "Call rows",
        "Unique call_id",
        "Duplicate call_id rows",
        "Missing call_id"
    ],
    "value": [
        len(calls_raw),
        calls_raw["call_id"].nunique(),
        calls_raw["call_id"].duplicated().sum(),
        calls_raw["call_id"].isna().sum()
    ]
})

call_key_summary

,metric,value
0,Call rows,91350
1,Unique call_id,90000
2,Duplicate call_id rows,1350
3,Missing call_id,0


In [16]:
#CALL ATTEMPT → CALL VALIDATION



valid_call_ids = set(
    calls_raw["call_id"]
    .dropna()
    .astype(str)
)

call_attempts_raw["call_id_match"] = (
    call_attempts_raw["call_id"]
    .astype("string")
    .isin(valid_call_ids)
)

attempt_validation = pd.DataFrame({
    "metric": [
        "Total call attempts",
        "Attempts linked to calls",
        "Attempts with unmatched call_id"
    ],
    "count": [
        len(call_attempts_raw),
        call_attempts_raw["call_id_match"].sum(),
        (~call_attempts_raw["call_id_match"]).sum()
    ]
})

attempt_validation["percentage"] = (
    attempt_validation["count"]
    / len(call_attempts_raw)
    * 100
)

attempt_validation


,metric,count,percentage
0,Total call attempts,120000,100.0
1,Attempts linked to calls,120000,100.0
2,Attempts with unmatched call_id,0,0.0


In [17]:
#ATTEMPTS PER CALL
attempts_per_call = (
    call_attempts_raw
    .groupby("call_id")
    .agg(
        attempt_count=("attempt_id", "nunique")
    )
    .reset_index()
)

print(
    "Calls represented in call_attempts:",
    len(attempts_per_call)
)

display(
    attempts_per_call["attempt_count"]
    .value_counts()
    .sort_index()
    .rename_axis("attempts_per_call")
    .reset_index(name="call_count")
)

Calls represented in call_attempts: 66244


,attempts_per_call,call_count
0,1,31497
1,2,21336
2,3,9208
3,4,3120
4,5,834
5,6,199
6,7,41
7,8,8
8,12,1


In [18]:
#CALL DISPOSITION → CALL VALIDATION



valid_call_ids = set(
    calls_raw["call_id"]
    .dropna()
    .astype(str)
)

call_dispositions_raw["call_id_match"] = (
    call_dispositions_raw["call_id"]
    .astype("string")
    .isin(valid_call_ids)
)

disposition_validation = pd.DataFrame({
    "metric": [
        "Disposition rows",
        "Dispositions linked to calls",
        "Dispositions with unmatched call_id"
    ],
    "count": [
        len(call_dispositions_raw),
        call_dispositions_raw["call_id_match"].sum(),
        (~call_dispositions_raw["call_id_match"]).sum()
    ]
})

disposition_validation["percentage"] = (
    disposition_validation["count"]
    / len(call_dispositions_raw)
    * 100
)

disposition_validation

,metric,count,percentage
0,Disposition rows,35000,100.0
1,Dispositions linked to calls,35000,100.0
2,Dispositions with unmatched call_id,0,0.0


In [19]:
#DISPOSITIONS PER CALL
dispositions_per_call = (
    call_dispositions_raw
    .groupby("call_id")
    .agg(
        disposition_count=("call_id", "size")
    )
    .reset_index()
)

display(
    dispositions_per_call["disposition_count"]
    .value_counts()
    .sort_index()
    .rename_axis("dispositions_per_call")
    .reset_index(name="call_count")
)

,dispositions_per_call,call_count
0,1,23689
1,2,4613
2,3,596
3,4,68
4,5,5


In [20]:
#GOLDEN CALLS — CALL-LEVEL GRAIN



golden_calls = calls_raw.copy()

# Parse event timestamp
golden_calls["event_at_dt"] = pd.to_datetime(
    golden_calls["event_at"],
    errors="coerce"
)

# Quality flag
golden_calls["quality_flag"] = "VALID"

# Missing call_id
golden_calls.loc[
    golden_calls["call_id"].isna(),
    "quality_flag"
] = "MISSING_CALL_ID"

# Duplicate call_id
duplicate_call_ids = golden_calls[
    golden_calls["call_id"].duplicated(keep=False)
]["call_id"]

golden_calls.loc[
    golden_calls["call_id"].isin(duplicate_call_ids)
    & golden_calls["call_id"].notna(),
    "quality_flag"
] = "DUPLICATE_CALL_ID"

print(
    golden_calls["quality_flag"]
    .value_counts(dropna=False)
)

print("\nGolden call rows:", len(golden_calls))


quality_flag
VALID                88650
DUPLICATE_CALL_ID     2700
Name: count, dtype: int64

Golden call rows: 91350


In [21]:
#CALL-LEVEL ATTEMPT SUMMARY
attempt_summary = (
    call_attempts_raw
    .groupby("call_id")
    .agg(
        attempt_count=("attempt_id", "nunique"),
        unique_attempt_agents=("agent_id", "nunique"),
        unique_attempt_vendors=("vendor_id", "nunique")
    )
    .reset_index()
)

display(attempt_summary.head(20))

,call_id,attempt_count,unique_attempt_agents,unique_attempt_vendors
0,CALL0000001,2,2,2
1,CALL0000004,1,1,1
2,CALL0000005,3,3,3
3,CALL0000006,2,2,2
4,CALL0000008,1,1,1
5,CALL0000010,1,1,1
6,CALL0000012,2,2,2
7,CALL0000013,1,1,1
8,CALL0000016,2,2,2
9,CALL0000017,2,2,2


In [22]:
#CALL-LEVEL DISPOSITION SUMMARY
disposition_summary = (
    call_dispositions_raw
    .groupby("call_id")
    .agg(
        disposition_records=("call_id", "size")
    )
    .reset_index()
)

display(disposition_summary.head(20))

,call_id,disposition_records
0,CALL0000004,1
1,CALL0000007,1
2,CALL0000009,1
3,CALL0000011,1
4,CALL0000012,2
5,CALL0000013,1
6,CALL0000016,2
7,CALL0000017,1
8,CALL0000021,1
9,CALL0000027,1


In [23]:
#ATTACH CHILD-TABLE SUMMARIES WITHOUT ROW MULTIPLICATION
golden_calls = golden_calls.merge(
    attempt_summary,
    on="call_id",
    how="left"
)

golden_calls = golden_calls.merge(
    disposition_summary,
    on="call_id",
    how="left"
)

print("Golden calls rows:", len(golden_calls))
print("Original calls rows:", len(calls_raw))

assert len(golden_calls) == len(calls_raw)

display(
    golden_calls.head(20)
)

Golden calls rows: 91350
Original calls rows: 91350


,call_id,account_id,borrower_id,event_at,agent_id,campaign_id,direction,vendor_id,call_status,duration_sec,timezone,event_at_dt,quality_flag,attempt_count,unique_attempt_agents,unique_attempt_vendors,disposition_records
0,CALL0000001,ACC0011505,BRW0007139,2026-07-15 15:36:22,AGT0000955,CMP0000015,OUTBOUND,VND0000013,NO_ANSWER,601,Asia/Dubai,2026-07-15 15:36:22,VALID,2.0,2.0,2.0,NaN
1,CALL0000002,ACC0002025,BRW0006253,2026-06-10 06:48:27,AGT0000853,CMP0000060,OUTBOUND,VND0000001,ANSWERED,224,Asia/Kolkata,2026-06-10 06:48:27,VALID,NaN,NaN,NaN,NaN
2,CALL0000003,ACC0013375,BRW0007663,2026-04-07 00:35:35,AGT0000525,CMP0000060,OUTBOUND,VND0000008,FAILED,7,Asia/Dubai,2026-04-07 00:35:35,VALID,NaN,NaN,NaN,NaN
3,CALL0000004,ACC0021534,BRW0008873,2026-02-12 14:16:57,AGT0000945,CMP0000060,OUTBOUND,VND0000001,VOICEMAIL,896,Asia/Dubai,2026-02-12 14:16:57,VALID,1.0,1.0,1.0,1.0
4,CALL0000005,ACC0018502,BRW0004996,2026-05-24 15:33:12,AGT0000966,CMP0000053,OUTBOUND,VND0000006,ANSWERED,711,Asia/Kolkata,2026-05-24 15:33:12,VALID,3.0,3.0,3.0,NaN
5,CALL0000006,ACC0011855,BRW0010726,2026-02-04 11:37:08,AGT0000864,CMP0000082,OUTBOUND,VND0000011,NO_ANSWER,18,Asia/Dubai,2026-02-04 11:37:08,VALID,2.0,2.0,2.0,NaN
6,CALL0000007,ACC0027278,BRW0005268,2026-02-10 10:59:05,AGT0000524,CMP0000001,OUTBOUND,VND0000009,VOICEMAIL,825,Asia/Kolkata,2026-02-10 10:59:05,VALID,NaN,NaN,NaN,1.0
7,CALL0000008,ACC0007144,BRW0001246,2026-06-14 23:37:56,AGT0000615,CMP0000115,OUTBOUND,VND0000003,VOICEMAIL,437,Asia/Dubai,2026-06-14 23:37:56,VALID,1.0,1.0,1.0,NaN
8,CALL0000009,ACC0003902,BRW0010076,2026-03-17 16:24:42,AGT0000585,CMP0000090,OUTBOUND,VND0000012,VOICEMAIL,318,UTC,2026-03-17 16:24:42,VALID,NaN,NaN,NaN,1.0
9,CALL0000010,ACC0009188,BRW0010503,2026-06-24 04:26:34,AGT0000238,CMP0000092,OUTBOUND,VND0000014,ANSWERED,260,UTC,2026-06-24 04:26:34,VALID,1.0,1.0,1.0,NaN


In [24]:
#SAVE GOLDEN CALLS
call_output = (
    PROCESSED_DIR / "golden_calls.csv"
)

golden_calls.to_csv(
    call_output,
    index=False
)

print("Saved:", call_output)
print("Rows:", len(golden_calls))


Saved: d:\OneDrive\Documents\CLASSROOM\collections-data-analytics\data\processed\golden_calls.csv
Rows: 91350


In [25]:
#LOAD REMAINING COLLECTION-EVENT TABLES
remaining_tables = [
    "promises_to_pay",
    "field_visits",
    "whatsapp_events",
    "sms_events",
    "daily_targeting",
    "campaigns",
    "agents",
    "vendor_telephony"
]

for table_name in remaining_tables:
    df = datasets[table_name]

    print(
        f"{table_name}: "
        f"{len(df):,} rows × {len(df.columns)} columns"
    )
    print("Columns:", df.columns.tolist())
    print()

promises_to_pay: 18,000 rows × 9 columns
Columns: ['ptp_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'promised_amount', 'promised_date', 'status', 'source']

field_visits: 25,000 rows × 10 columns
Columns: ['visit_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'visit_type', 'outcome', 'latitude', 'longitude', 'scheduled_at']

whatsapp_events: 60,600 rows × 8 columns
Columns: ['whatsapp_event_id', 'account_id', 'borrower_id', 'event_at', 'message_id', 'event_type', 'template_code', 'provider_id']

sms_events: 45,000 rows × 8 columns
Columns: ['sms_event_id', 'account_id', 'borrower_id', 'event_at', 'message_id', 'event_type', 'template_code', 'provider_id']

daily_targeting: 45,000 rows × 7 columns
Columns: ['target_id', 'account_id', 'campaign_id', 'target_date', 'priority', 'recommended_channel', 'status']

campaigns: 120 rows × 7 columns
Columns: ['campaign_id', 'campaign_name', 'channel', 'strategy_version', 'start_at', 'target_definition', 'end_at']

agents:

In [26]:
#EVENT TABLE KEY INVENTORY
event_key_inventory = []

for table_name in remaining_tables:

    df = datasets[table_name]

    for col in df.columns:

        if (
            col.lower().endswith("_id")
            or col.lower() == "id"
        ):
            event_key_inventory.append({
                "dataset": table_name,
                "column": col,
                "rows": len(df),
                "unique_values": df[col].nunique(dropna=True),
                "missing_values": df[col].isna().sum(),
                "duplicate_values": df[col].duplicated().sum()
            })

event_key_inventory_df = pd.DataFrame(
    event_key_inventory
)

event_key_inventory_df

,dataset,column,rows,unique_values,missing_values,duplicate_values
0,promises_to_pay,ptp_id,18000,18000,0,0
1,promises_to_pay,account_id,18000,13532,0,4468
2,promises_to_pay,borrower_id,18000,9299,0,8701
3,promises_to_pay,agent_id,18000,1000,0,17000
4,field_visits,visit_id,25000,25000,0,0
5,field_visits,account_id,25000,16908,0,8092
6,field_visits,borrower_id,25000,10537,0,14463
7,field_visits,agent_id,25000,1000,0,24000
8,whatsapp_events,whatsapp_event_id,60600,60000,0,600
9,whatsapp_events,account_id,60600,25924,0,34676


In [27]:
# EVENT → ACCOUNT LINKAGE



valid_account_ids = set(
    accounts_raw["account_id"]
    .dropna()
    .astype(str)
)

account_linkage_results = []

for table_name in remaining_tables:

    df = datasets[table_name]

    if "account_id" not in df.columns:
        continue

    account_ids = (
        df["account_id"]
        .dropna()
        .astype(str)
    )

    matched = account_ids.isin(
        valid_account_ids
    )

    account_linkage_results.append({
        "dataset": table_name,
        "rows_with_account_id": len(account_ids),
        "matched_account_ids": matched.sum(),
        "unmatched_account_ids": (~matched).sum(),
        "unmatched_pct": round(
            (~matched).mean() * 100,
            2
        ) if len(account_ids) else 0
    })

account_linkage_df = pd.DataFrame(
    account_linkage_results
)

account_linkage_df


,dataset,rows_with_account_id,matched_account_ids,unmatched_account_ids,unmatched_pct
0,promises_to_pay,18000,18000,0,0.0
1,field_visits,25000,25000,0,0.0
2,whatsapp_events,60600,60600,0,0.0
3,sms_events,45000,45000,0,0.0
4,daily_targeting,45000,45000,0,0.0


In [28]:

#  EVENT QUALITY FLAGS


golden_event_tables = {}

for table_name in remaining_tables:

    df = datasets[table_name].copy()

    df["quality_flag"] = "VALID"

    if "account_id" in df.columns:

        unmatched_mask = (
            df["account_id"].notna()
            & ~df["account_id"]
                .astype(str)
                .isin(valid_account_ids)
        )

        df.loc[
            unmatched_mask,
            "quality_flag"
        ] = "UNMATCHED_ACCOUNT_ID"

    golden_event_tables[table_name] = df

    print(
        f"\n{table_name}"
    )

    print(
        df["quality_flag"]
        .value_counts(dropna=False)
    )


promises_to_pay
quality_flag
VALID    18000
Name: count, dtype: int64

field_visits
quality_flag
VALID    25000
Name: count, dtype: int64

whatsapp_events
quality_flag
VALID    60600
Name: count, dtype: int64

sms_events
quality_flag
VALID    45000
Name: count, dtype: int64

daily_targeting
quality_flag
VALID    45000
Name: count, dtype: int64

campaigns
quality_flag
VALID    120
Name: count, dtype: int64

agents
quality_flag
VALID    30000
Name: count, dtype: int64

vendor_telephony
quality_flag
VALID    15
Name: count, dtype: int64


In [29]:

#  SAVE GOLDEN EVENT TABLES


for table_name, df in golden_event_tables.items():

    output_file = (
        PROCESSED_DIR
        / f"golden_{table_name}.csv"
    )

    df.to_csv(
        output_file,
        index=False
    )

    print(
        f"Saved {table_name}: {len(df):,} rows"
    )

Saved promises_to_pay: 18,000 rows
Saved field_visits: 25,000 rows
Saved whatsapp_events: 60,600 rows
Saved sms_events: 45,000 rows
Saved daily_targeting: 45,000 rows
Saved campaigns: 120 rows
Saved agents: 30,000 rows
Saved vendor_telephony: 15 rows


# GOLDEN DATASET — METHODOLOGY & ASSUMPTIONS

## Source-of-Truth Decisions

- `accounts` is treated as the account-level source of truth for outstanding
  amount, DPD, risk segment, loan type, and current account attributes.
- `payments` is treated as the source for successful payment events and
  recovered amount.
- `account_status_history` is retained as the historical source for
  time-dependent account status.
- Child interaction tables are retained at their native grain and summarized
  before being joined to account/call-level tables to prevent row
  multiplication.

## Entity Resolution

Account-to-borrower relationships are validated by matching `accounts.borrower_id`
against the available borrower identifiers.

Unmatched borrower relationships are flagged as `UNRESOLVED_BORROWER_ID`.
Accounts are preserved rather than removed because an unresolved relationship
does not by itself invalidate the account.

Event tables containing `account_id` are similarly checked against the valid
account population. Unmatched account relationships are flagged rather than
silently dropped.

## Deduplication Logic

Exact duplicate payment rows are identified using full-row duplicate detection
and excluded from the payment golden dataset.

Repeated payment identifiers or references are not automatically deduplicated
because repeated identifiers do not necessarily prove that payment events are
duplicates. Business-key deduplication requires additional validation.

For calls, duplicate `call_id` values are flagged for investigation rather than
silently deleting records.

## Missing-Data Treatment

Missing identifiers are explicitly flagged where they affect entity linkage.

Missing payment references are classified as `MISSING_REFERENCE`.

Missing or unresolved relationships are preserved where possible so that the
scale of the data-quality issue remains visible.

No unsupported imputation is applied to business-critical identifiers.

## Timestamp Treatment

Event timestamps are parsed using `pandas.to_datetime(..., errors="coerce")`
before historical comparisons.

Invalid timestamps therefore become missing datetime values rather than
causing silent parsing failures.

The current golden-dataset pipeline preserves the source timestamp values and
creates parsed datetime fields where required for analysis.

## Payment Attribution

Successful recovery is based on payment records where
`payment_status = 'SUCCESS'`.

Payments are not attributed to campaigns, calls, or other interactions using
an unvalidated latest-touch rule.

Campaign/channel attribution therefore remains subject to the attribution
limitations documented in the analysis.

## Historical Changes

`account_status_history` is retained separately from the current account
status.

For reconciliation, historical records are ordered by `account_id`,
`event_at`, and `recorded_at`, and the latest valid historical status is
compared with the current account status.

Mismatches are flagged as `STATUS_MISMATCH` rather than overwriting historical
information.

## Exclusion Rules

Records are excluded from the payment golden dataset only when they are
identified as exact duplicate rows.

Unmatched entities and other quality anomalies are generally retained and
flagged unless there is sufficient evidence that the record is invalid.

This approach minimizes unsupported data loss.

## Data-Quality Issues

The golden-data process explicitly checks:

- Duplicate identifiers and duplicate rows
- Missing identifiers
- Unmatched borrower relationships
- Unmatched account relationships
- Call and child-table linkage
- Current versus historical account-status mismatches
- Payment-reference quality

Detailed findings are documented separately in the Data Quality Report.

## Assumptions

- A successful payment record represents a recovery event unless identified
  as an exact duplicate.
- Repeated identifiers are treated as anomalies requiring investigation, not
  automatic duplicates.
- Historical account status should not be replaced by the current status.
- Native event-table grain should be preserved wherever possible.
- No causal conclusion is made solely from observed associations.

## Cleaning Impact

The pipeline records raw and golden row counts for the major analytical tables
and explicitly reports excluded exact duplicate payment records.

Unresolved relationships and status mismatches are retained as quality flags,
so their impact remains measurable rather than being hidden through deletion.

Where the available analysis does not provide a validated financial impact,
no monetary impact is invented.

## Golden Dataset Principle

The golden layer prioritizes **traceability, conservative treatment, and
reproducibility** over aggressive cleaning.

Raw records → Quality checks → Flagged/corrected records → Golden analytical
datasets